# Segmented bank line visualization

**Goal:** explore whether splitting a scope region into N along-centerline segments
gives a more representative picture of bank position than a single global mean.

**Part 1** — reproduce the existing visualization (1 offset line per year per region).

**Part 2** — new segmented visualization (N offset sub-segments per year per region).

In [ ]:
import os, sys
from pathlib import Path

_cwd = Path.cwd()
for _c in [_cwd, *_cwd.parents]:
    if (_c / 'src').exists():
        _backend = _c
        break
else:
    raise RuntimeError('Cannot find backend root')

os.chdir(_backend)
if str(_backend) not in sys.path:
    sys.path.insert(0, str(_backend))

print('backend root:', _backend)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import geopandas as gpd

import src.paths as PATHS
from src.erosion.centerline_utils import (
    ensure_location_id_column,
    compute_qualifying_regions,
    pick_across_clusters,
)
from src.erosion.plot_utils import (
    draw_region,
    draw_region_segmented,
    make_legend_handles,
    YEAR_COLORS,
)

RAW_GPKG = PATHS.DATA_DIR / '01_raw/erosion/wocu_output_fase2_20260210.gpkg'

print('Imports OK')

## 1. Load data

In [ ]:
print('Loading bank points ...')
bank_points = ensure_location_id_column(gpd.read_file(RAW_GPKG, layer='punten_oever'))
centerlines = ensure_location_id_column(gpd.read_file(RAW_GPKG, layer='centrelines'))
scope       = ensure_location_id_column(gpd.read_file(RAW_GPKG, layer='vlakken_scope'))

cl_lookup    = centerlines.set_index('location_id')['geometry'].to_dict()
scope_lookup = scope.set_index('location_id')['geometry'].to_dict()

print(f'Bank points : {len(bank_points):,}')
print(f'Centerlines : {len(cl_lookup):,}')
print(f'Scope regions: {len(scope_lookup):,}')

## 2. Pick interesting scope regions

Use `compute_qualifying_regions` to find regions with noticeable erosion across all 3 timestamps.
We pick a few spread across clusters.

In [ ]:
qualifying = compute_qualifying_regions(bank_points, min_shift=3.0, n_points=3, n_timepoints=3)
print(f'Qualifying regions (shift ≥ 3m at both intervals): {len(qualifying):,}')
qualifying.head(10)

In [ ]:
# Pick 4 interesting regions spread across clusters, with clear erosion signal
candidates = qualifying.index.tolist()
SELECTED = pick_across_clusters(candidates, n=4)

print('Selected regions:')
for loc in SELECTED:
    row = qualifying.loc[loc]
    print(f'  {loc:<35}  shift t1→t2={row["shift_t1t2"]:.1f}m  t2→t3={row["shift_t2t3"]:.1f}m')

## Part 1 — Existing visualization (1 offset line per year)

One mean dist from the top-3 furthest OK points → one parallel offset line per year.

In [ ]:
n = len(SELECTED)
fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 8), constrained_layout=True)
if n == 1:
    axes = [axes]

for col, loc_id in enumerate(SELECTED):
    draw_region(
        axes[col], loc_id,
        cl_lookup=cl_lookup,
        scope_lookup=scope_lookup,
        bank_points=bank_points,
        n_points=3,
        col_idx=col,
    )

fig.legend(
    handles=make_legend_handles(n_points=3),
    loc='lower center', ncol=6, fontsize=8,
    bbox_to_anchor=(0.5, -0.06), frameon=True,
)
fig.suptitle('Part 1 — single offset line per year (top-3 furthest OK points, global mean)', fontsize=11)
plt.show()

## Part 2 — Segmented visualization

The centerline is divided into N equal-length sub-segments.  
Bank points are assigned to the sub-segment they project onto.  
Each sub-segment gets its own independent offset line per year.

Try different `N_SEGMENTS` values to see where the resolution gain starts to outweigh
the noise from sparse points per segment.

In [ ]:
N_SEGMENTS = 10   # ← try 5, 10, 20
N_POINTS   = 1    # furthest N OK points per sub-segment per year

n = len(SELECTED)
fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 8), constrained_layout=True)
if n == 1:
    axes = [axes]

for col, loc_id in enumerate(SELECTED):
    draw_region_segmented(
        axes[col], loc_id,
        cl_lookup=cl_lookup,
        scope_lookup=scope_lookup,
        bank_points=bank_points,
        n_segments=N_SEGMENTS,
        n_points=N_POINTS,
        col_idx=col,
    )

fig.suptitle(
    f'Part 2 — {N_SEGMENTS} segments × top-{N_POINTS} furthest OK points per segment',
    fontsize=11,
)
plt.show()

## Part 3 — Side-by-side comparison for one region

Pick a single region and compare different numbers of segments directly.

In [ ]:
FOCUS_REGION = SELECTED[0]   # ← change to any location_id
SEGMENT_OPTIONS = [1, 5, 10, 20]

fig, axes = plt.subplots(1, len(SEGMENT_OPTIONS), figsize=(5.5 * len(SEGMENT_OPTIONS), 8),
                          constrained_layout=True)

for col, n_seg in enumerate(SEGMENT_OPTIONS):
    draw_region_segmented(
        axes[col], FOCUS_REGION,
        cl_lookup=cl_lookup,
        scope_lookup=scope_lookup,
        bank_points=bank_points,
        n_segments=n_seg,
        n_points=1,
        col_idx=col,
    )
    axes[col].set_title(f'{n_seg} segment{"s" if n_seg > 1 else ""}\n{FOCUS_REGION}',
                        fontsize=8)

fig.suptitle(f'Effect of N segments — {FOCUS_REGION}', fontsize=11)
plt.show()

## Part 4 — Point density per segment

Check how many OK points fall in each segment across years — to ensure segments are not too sparse.

In [ ]:
import pandas as pd
import numpy as np

N_SEGMENTS = 10

fig, axes = plt.subplots(1, len(SELECTED), figsize=(4 * len(SELECTED), 3),
                          constrained_layout=True, sharey=True)
if len(SELECTED) == 1:
    axes = [axes]

for ax, loc_id in zip(axes, SELECTED):
    cline = cl_lookup.get(loc_id)
    grp = bank_points[(bank_points['location_id'] == loc_id) & (bank_points['status'] == 'OK')].copy()
    if cline is None or grp.empty:
        ax.set_title(f'{loc_id}\n(no data)')
        continue

    grp['seg'] = (
        grp.geometry.apply(lambda pt: cline.project(pt, normalized=True))
        * N_SEGMENTS
    ).astype(int).clip(0, N_SEGMENTS - 1)

    counts = grp.groupby(['dtm_date', 'seg']).size().unstack(fill_value=0)
    x = np.arange(N_SEGMENTS)
    width = 0.8 / len(counts)
    for i, (date, row) in enumerate(counts.iterrows()):
        color = YEAR_COLORS[i % len(YEAR_COLORS)]
        vals = [row.get(s, 0) for s in range(N_SEGMENTS)]
        ax.bar(x + i * width, vals, width, label=str(date), color=color, alpha=0.8)

    ax.set_title(loc_id, fontsize=7)
    ax.set_xlabel('Segment index')
    ax.set_xticks(x + width)
    ax.set_xticklabels(x, fontsize=7)
    ax.legend(fontsize=6)

axes[0].set_ylabel('OK point count')
fig.suptitle(f'OK point count per segment (N={N_SEGMENTS}) per year', fontsize=10)
plt.show()